# Notebook 6 - Computer Vision

Roadmap role: Week 6. This Colab workflow preserves dataset provenance, official splits, model settings, raw outputs, and negative results. Agriculture-Vision is a semantic-segmentation benchmark; generic YOLO inference here is only the roadmap-required smoke-test baseline.

## 1. Clone The Week 6 Branch

Run this in a fresh Colab runtime. Licensed imagery and large checkpoints persist in the operator's private Google Drive; source code and nonrestricted reproducibility records remain in GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path('/content/shepherd-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/week6-vision', 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
os.chdir(REPO)
print('repository', Path.cwd())
print('commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Record The Runtime

Dataset acquisition, layout inspection, manifest construction, and label auditing may run in CPU debug mode. YOLO inference and future segmentation training remain T4-gated. A CPU debug run is not a model-performance experiment.

In [ ]:
import torch
CUDA_DEVICES = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
HAS_T4 = any('T4' in name for name in CUDA_DEVICES)
RUN_MODE = 't4_experiment' if HAS_T4 else 'cpu_debug'
print('run_mode', RUN_MODE)
print('cuda_devices', CUDA_DEVICES)
print('torch_version', torch.__version__)


## 3. Review And Accept Agriculture-Vision Terms

Official terms: https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_paper_2020/Agriculture-Vision%20Dataset%20Terms%20of%20Use.pdf

Downloading signifies agreement. The terms permit limited non-commercial research use and prohibit redistribution. Only continue if you personally reviewed and accept them.

In [ ]:
TERMS_ACKNOWLEDGMENT = input('Type I ACCEPT AGRICULTURE-VISION TERMS after reviewing them: ').strip()
if TERMS_ACKNOWLEDGMENT != 'I ACCEPT AGRICULTURE-VISION TERMS':
    raise RuntimeError('Terms were not accepted; dataset acquisition stopped.')
print('Terms acknowledged for this Colab session.')


## 4. Mount The Private Google Drive Cache

The Agriculture-Vision terms prohibit redistribution, so the licensed dataset must not be committed to the public repository. This cell creates a private Drive cache that survives Colab runtime recycling and links it into the repository's expected dataset path.

In [ ]:
from google.colab import drive

if TERMS_ACKNOWLEDGMENT != 'I ACCEPT AGRICULTURE-VISION TERMS':
    raise RuntimeError('Terms acknowledgment is required in this session.')
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/shepherd-ai-private/week6')
DATASET_DIR = DRIVE_ROOT / 'agriculture-vision'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
REPO_DATASET_DIR = REPO / 'datasets' / 'aerial_images' / 'agriculture-vision'
if REPO_DATASET_DIR.is_symlink():
    REPO_DATASET_DIR.unlink()
elif REPO_DATASET_DIR.exists():
    if any(REPO_DATASET_DIR.iterdir()):
        raise RuntimeError(f'Refusing to replace non-empty runtime dataset directory: {REPO_DATASET_DIR}')
    REPO_DATASET_DIR.rmdir()
REPO_DATASET_DIR.symlink_to(DATASET_DIR, target_is_directory=True)
print('private_drive_cache', DRIVE_ROOT)
print('repository_dataset_link', REPO_DATASET_DIR, '->', REPO_DATASET_DIR.resolve())


## 5. Cache And Extract The Official 2017 Archive

The archive is approximately 1.88 GB. It is downloaded from the official IntelinAir AWS bucket only when absent from Drive. Its SHA-256 is recalculated every session and written to a non-image provenance record that may be committed to GitHub.

In [ ]:
import hashlib
import json
import tarfile
from urllib.request import urlretrieve

BASE = 'https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_paper_2020/Dataset'
ARCHIVE = DATASET_DIR / 'data2017_miniscale.tar.gz'
SPLITS = DATASET_DIR / 'data2017_splits.json'
if not ARCHIVE.exists():
    urlretrieve(f'{BASE}/data2017_miniscale.tar.gz', ARCHIVE)
if not SPLITS.exists():
    urlretrieve(f'{BASE}/data2017_splits.json', SPLITS)
digest = hashlib.sha256()
with ARCHIVE.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
ARCHIVE_SHA256 = digest.hexdigest()
print('archive_sha256', ARCHIVE_SHA256)
EXTRACTED = DATASET_DIR / 'data2017'
EXTRACTED.mkdir(exist_ok=True)
EXPECTED_RGB_TILES = 8345
rgb_tiles = list(EXTRACTED.rglob('field_images/rgb/*.jpg'))
if len(rgb_tiles) != EXPECTED_RGB_TILES:
    print(f'Repairing incomplete extraction: found {len(rgb_tiles)} of {EXPECTED_RGB_TILES} RGB tiles')
    with tarfile.open(ARCHIVE, 'r:gz') as archive:
        archive.extractall(EXTRACTED, filter='data')
    rgb_tiles = list(EXTRACTED.rglob('field_images/rgb/*.jpg'))
if len(rgb_tiles) != EXPECTED_RGB_TILES:
    raise RuntimeError(f'Extraction validation failed: found {len(rgb_tiles)} of {EXPECTED_RGB_TILES} RGB tiles')
print('rgb_tiles', len(rgb_tiles))
print('extracted_to', EXTRACTED)
PROVENANCE = REPO / 'outputs' / 'evaluations' / 'week6_agriculture_vision_cache_provenance.json'
PROVENANCE.parent.mkdir(parents=True, exist_ok=True)
PROVENANCE.write_text(json.dumps({
    'source_url': f'{BASE}/data2017_miniscale.tar.gz',
    'archive_filename': ARCHIVE.name,
    'archive_sha256': ARCHIVE_SHA256,
    'split_filename': SPLITS.name,
    'storage': 'private_google_drive_cache',
    'licensed_pixels_committed': False,
}, indent=2) + '\n', encoding='utf-8')
print('provenance_record', PROVENANCE)


## 6. Record The Extracted Dataset Layout

Record relative paths and counts before implementing a label loader. This is the evidence for the 2017 class-mask and valid-region mapping; it does not copy licensed pixels or infer label semantics.

In [ ]:
!python scripts/inspect_agriculture_vision_layout.py --dataset-dir datasets/aerial_images/agriculture-vision/data2017 --output outputs/evaluations/week6_agriculture_vision_layout.json --sample-limit 120


## 7. Build And Validate A Leakage-Safe Subset Manifest

The official farmland-level split JSON is authoritative. Selection is deterministic and limited to 10 RGB images per split. Every selected image receives a SHA-256 digest.

In [ ]:
subprocess.run(['python', 'scripts/prepare_agriculture_vision_subset.py', '--dataset-dir', str(EXTRACTED), '--dataset-root', str(DRIVE_ROOT), '--split-json', str(SPLITS), '--output', 'datasets/aerial_images/manifest.jsonl', '--max-per-split', '10', '--accept-terms'], check=True)
subprocess.run(['python', 'scripts/validate_vision_manifest.py', '--manifest', 'datasets/aerial_images/manifest.jsonl', '--dataset-root', str(DRIVE_ROOT), '--summary-output', 'outputs/evaluations/week6_vision_manifest_summary.json'], check=True)


## 8. Audit Train And Validation Labels

Validate aligned masks and record class prevalence and overlap before training. This command rejects the test split so final evaluation labels remain untouched.

In [ ]:
subprocess.run(['python', 'scripts/validate_agriculture_vision_labels.py', '--manifest', 'datasets/aerial_images/manifest.jsonl', '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--split', 'train', '--split', 'validation', '--output', 'outputs/evaluations/week6_agriculture_vision_label_audit.json'], check=True)


## 9. Run The YOLO Smoke-Test Baseline

Detection counts and confidence values are pipeline outputs, not Agriculture-Vision anomaly performance. Zero detections are preserved as a valid negative result.

In [ ]:
if not HAS_T4:
    raise RuntimeError('YOLO inference requires a T4. CPU debug steps 1-8 are still valid; resume this cell when T4 quota is available.')
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[vision]'], check=True)
subprocess.run(['python', 'scripts/run_yolo_detection.py', '--manifest', 'datasets/aerial_images/manifest.jsonl', '--dataset-root', str(DRIVE_ROOT), '--model', 'yolov8n.pt', '--confidence', '0.25', '--device', '0', '--required-device-substring', 'T4', '--predictions-output', 'outputs/evaluations/week6_yolo_detections.jsonl', '--summary-output', 'outputs/evaluations/week6_yolo_detection_summary.json', '--annotated-dir', 'outputs/visualizations/week6_yolo'], check=True)
